# part1.ipynb — Baseline classifier

Trains DistilBERT and writes `artifacts/baseline_distilbert/`.


### i220524 FAST-NUCES Assignment 2 — Responsible & Explainable AI


## Environment setup

Install dependencies (local or Colab), then continue. On Colab: **Runtime → Change runtime type → GPU**.



In [ ]:
"""Install dependencies via `pip install -r requirements.txt` in a fresh environment (e.g. Colab)."""


def ensure_pkg():
    """No-op placeholder; run pip manually or automate here if desired."""
    return None


ensure_pkg()


In [ ]:
"""Imports and global configuration for reproducibility.

Sets USE_TF=0 before HuggingFace loads, and forces trainer_utils.is_tf_available() to False so Trainer.setup never imports TensorFlow (broken TF wheels still report as installed).
"""
import os
os.environ["USE_TF"] = "0"
import re
import random
import zipfile
from typing import Any, Dict, List, Tuple

import joblib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from sklearn.base import clone

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import transformers.trainer_utils as _hf_trainer_utils
_hf_trainer_utils.is_tf_available = lambda: False
from datasets import Dataset

from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import Reweighing
from aif360.metrics import ClassificationMetric

sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ZIP_PATH = os.path.join(os.getcwd(), "jigsaw-unintended-bias-in-toxicity-classification.zip")
TRAIN_CSV_NAME = "train.csv"
ARTIFACT_DIR = os.path.join(os.getcwd(), "artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

BASELINE_DIR = os.path.join(ARTIFACT_DIR, "baseline_distilbert")
POISON_DIR = os.path.join(ARTIFACT_DIR, "poisoned_distilbert")
REWEIGHT_DIR = os.path.join(ARTIFACT_DIR, "reweighted_distilbert")
OVERSAMPLE_DIR = os.path.join(ARTIFACT_DIR, "oversample_distilbert")
ISOTONIC_PATH = os.path.join(ARTIFACT_DIR, "isotonic_calibrator.joblib")

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
NUM_EPOCHS = 3
BATCH_SIZE = 16
LR = 2e-5

"""Optional: set True only for quick smoke tests (not for submission)."""
FAST_DEBUG = False
if FAST_DEBUG:
    NUM_EPOCHS = 1


## Load data from the zip (stratified 100k train / 20k eval)

We read only the columns needed to limit memory. **Label:** `y = (target >= 0.5).astype(int)`. `train_test_split` returns `(larger_slice, test_size_slice)`; the 20k holdout is the **second** return value.



In [ ]:
"""Load `train.csv` from the competition zip without extracting the full archive to disk."""


def read_train_from_zip(zip_path: str, csv_name: str) -> pd.DataFrame:
    """Read selected columns for memory efficiency and normalize names."""
    usecols = [
        "comment_text",
        "target",
        "black",
        "white",
        "muslim",
        "jewish",
        "homosexual_gay_or_lesbian",
    ]
    with zipfile.ZipFile(zip_path, "r") as zf:
        with zf.open(csv_name) as f:
            df = pd.read_csv(f, usecols=usecols)
    df = df.rename(columns={"target": "toxic"})
    df["comment_text"] = df["comment_text"].fillna("").astype(str)
    for c in ["toxic", "black", "white", "muslim", "jewish", "homosexual_gay_or_lesbian"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
    df["y"] = (df["toxic"] >= 0.5).astype(int)
    return df


df_all = read_train_from_zip(ZIP_PATH, TRAIN_CSV_NAME)
if FAST_DEBUG:
    eval_n = 2000
    train_n = 5000
    cap = eval_n + train_n + 5000
    df_all, _ = train_test_split(
        df_all, train_size=min(cap, len(df_all)), stratify=df_all["y"], random_state=SEED
    )
else:
    eval_n = 20000
    train_n = 100000

y = df_all["y"].values
train_remaining, eval_df = train_test_split(
    df_all, test_size=eval_n, stratify=y, random_state=SEED
)
y_rem = train_remaining["y"].values
train_df, _discard = train_test_split(
    train_remaining, train_size=train_n, stratify=y_rem, random_state=SEED
)

print("train_df:", train_df.shape, "eval_df:", eval_df.shape)
print("toxic rate train:", train_df["y"].mean(), "eval:", eval_df["y"].mean())


# Part 1 — Baseline DistilBERT classifier

We fine-tune `distilbert-base-uncased` with HuggingFace `Trainer` (cross-entropy). Tokenization: `max_length=128`, `truncation=True`.



In [ ]:
"""Fine-tune DistilBERT using HuggingFace Trainer (Part 1 core)."""
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_batch(examples, tokenizer_obj, max_length: int):
    """Tokenize comment batches for DistilBERT."""
    return tokenizer_obj(
        examples["comment_text"],
        truncation=True,
        max_length=max_length,
    )


train_ds = Dataset.from_pandas(train_df[["comment_text", "y"]].rename(columns={"y": "labels"}))
eval_ds = Dataset.from_pandas(eval_df[["comment_text", "y"]].rename(columns={"y": "labels"}))

train_ds = train_ds.map(
    lambda b: tokenize_batch(b, tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=["comment_text"],
)
eval_ds = eval_ds.map(
    lambda b: tokenize_batch(b, tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=["comment_text"],
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)


def compute_metrics(eval_pred):
    """Compute accuracy, macro-F1, binary toxic F1, and AUC when defined."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    pr = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    out = {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_binary": f1_score(labels, preds, pos_label=1),
    }
    try:
        out["auc"] = roc_auc_score(labels, pr)
    except ValueError:
        out["auc"] = float("nan")
    return out


args = TrainingArguments(
    output_dir=os.path.join(ARTIFACT_DIR, "trainer_baseline"),
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_steps=200,
    load_best_model_at_end=False,
    report_to=[],
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
trainer.save_model(BASELINE_DIR)
tokenizer.save_pretrained(BASELINE_DIR)
print(train_result)


In [ ]:
"""Baseline evaluation: metrics, ROC/PR curves, confusion matrix, threshold sweep."""
model.eval()
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

def hf_predict_proba_toxic(texts: List[str], m, tok, device_obj, max_len: int) -> np.ndarray:
    """Return toxic-class probabilities for a list of strings."""
    enc = tok(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    ).to(device_obj)
    with torch.no_grad():
        logits = m(**enc).logits
        pr = torch.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()
    return pr


texts_eval = eval_df["comment_text"].tolist()
y_true = eval_df["y"].to_numpy()
scores = hf_predict_proba_toxic(texts_eval, model, tokenizer, device, MAX_LENGTH)
y_pred_05 = (scores >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_true, y_pred_05))
print("F1 macro:", f1_score(y_true, y_pred_05, average="macro"))
print("F1 toxic (binary):", f1_score(y_true, y_pred_05, pos_label=1))
print("AUC-ROC:", roc_auc_score(y_true, scores))

cm = confusion_matrix(y_true, y_pred_05)
print("Confusion matrix (thr=0.5):", cm)

fpr, tpr, thr_roc = roc_curve(y_true, scores)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label="ROC")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC — baseline")
plt.legend()
plt.show()

prec, rec, pr_thr = precision_recall_curve(y_true, scores)
plt.figure(figsize=(6, 5))
plt.plot(rec, prec)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall — baseline")
plt.show()

rows = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_hat = (scores >= t).astype(int)
    rows.append(
        {
            "threshold": t,
            "f1_macro": f1_score(y_true, y_hat, average="macro"),
            "f1_toxic": f1_score(y_true, y_hat, pos_label=1),
        }
    )
thr_tbl = pd.DataFrame(rows)
print(thr_tbl)

CHOSEN_THRESHOLD = 0.5
print("Chosen threshold for later parts:", CHOSEN_THRESHOLD)


### Part 1 — Threshold justification (model answer — align with your F1 sweep table)

**Choice:** **0.5** is a reasonable default: it matches the dataset’s toxicity binarization (≥0.5) and avoids picking an extreme without a clear policy reason. If your sweep shows **meaningfully higher macro-F1 or toxic-class F1** at e.g. **0.4** or **0.6**, you can adopt that threshold and cite the table.

**Trade-off:** **Lower τ** → more true positives (more abuse caught) but more **false positives** (innocent users limited or removed). **Higher τ** → fewer false positives but more **false negatives** (harmful content stays up). There is no universal optimum—only a **harm–error mix** the product and legal team can defend.

**Platform stance (example):** If the org prioritizes **user trust and false-accusation risk**, bias toward a **slightly higher** threshold or route borderline scores to **human review** (Part 5). If it prioritizes **minimizing visible abuse**, bias **lower** and accept more review load and false-positive complaints. **Edit:** Add **one sentence** naming your best τ from the printed sweep and which priority you chose.

